# 🤡 Worst Trades — ALL‑TIME (Ridiculously Silly Edition)

Welcome to the **Hall of Oof**. This notebook:
- Calculates **all‑time** trade damage (season of the trade *after the trade week* **and** every season after).
- **Excludes** trades with **unrealized picks** (no crystal balls today).
- Shows **Top‑K** disasters + **per‑season** damage.
- NEW: **Asset Breakdown** — exactly who/what was traded, what each pick became, and **how many points each asset cost** the losing side.

> Pro tip: hydrate before reading; the salt content may be high 🧂.


## 🛠️ Pick a League & How Many Oofs

In [0]:
dbutils.widgets.text("league_name", "League of Inches", "League Name")
dbutils.widgets.text("top_k", "3", "Top K Trades")
league_name = dbutils.widgets.get("league_name")
top_k = int(dbutils.widgets.get("top_k"))
print(f"League: {league_name} | Top K: {top_k}")

## 🧮 Top Worst Trades — ALL‑TIME (Net Impact)

In [0]:
spark.sql("USE CATALOG workspace")

qt = """
WITH league_filter AS (
  SELECT league_id, name
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
  WHERE name = '__LEAGUE_NAME__'
),
tx AS (
  SELECT DISTINCT t.league_id, t.transaction_id
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN league_filter USING (league_id)
  WHERE t.type = 'trade'
),
tx_picks AS (
  SELECT b.league_id, b.transaction_id, COUNT(*) AS picks_expected
  FROM sleeper_trades.bridge_trade_pick_mapping_complete b
  JOIN tx USING (league_id, transaction_id)
  GROUP BY b.league_id, b.transaction_id
),
tx_picks_realized AS (
  SELECT r.league_id, r.transaction_id, COUNT(*) AS picks_realized
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id
),
tx_fully_realized AS (
  SELECT t.league_id, t.transaction_id
  FROM tx t
  LEFT JOIN tx_picks p USING (league_id, transaction_id)
  LEFT JOIN tx_picks_realized pr USING (league_id, transaction_id)
  WHERE COALESCE(p.picks_expected, 0) = COALESCE(pr.picks_realized, 0)
),
incoming_players AS (
  SELECT a.league_id, a.transaction_id, a.side_roster_id,
         st.season AS trade_season, st.week AS trade_week, a.player_id
  FROM sleeper_trades.fact_trade_assets_player_only a
  JOIN sleeper_trades.stg_trade_transactions st
    ON a.league_id=st.league_id AND a.transaction_id=st.transaction_id
  JOIN tx_fully_realized USING (league_id, transaction_id)
  WHERE a.direction='incoming'
),
post_player_points_side AS (
  SELECT ip.league_id, ip.transaction_id, ip.side_roster_id,
         fpw.season,
         SUM(fpw.points) AS player_points_side
  FROM incoming_players ip
  JOIN sleeper_core.fact_player_week fpw
    ON fpw.league_id=ip.league_id AND fpw.player_id=ip.player_id AND fpw.roster_id=ip.side_roster_id
   AND ( (fpw.season = ip.trade_season AND fpw.week > ip.trade_week) OR (CAST(fpw.season AS INT) > CAST(ip.trade_season AS INT)) )
  GROUP BY ip.league_id, ip.transaction_id, ip.side_roster_id, fpw.season
),
post_player_points_all AS (
  SELECT ip.league_id, ip.transaction_id, fpw.season,
         SUM(fpw.points) AS player_points_all
  FROM incoming_players ip
  JOIN sleeper_core.fact_player_week fpw
    ON fpw.league_id=ip.league_id AND fpw.player_id=ip.player_id
   AND ( (fpw.season = ip.trade_season AND fpw.week > ip.trade_week) OR (CAST(fpw.season AS INT) > CAST(ip.trade_season AS INT)) )
  GROUP BY ip.league_id, ip.transaction_id, fpw.season
),
pick_points_side AS (
  SELECT r.league_id, r.transaction_id, r.side_roster_id, r.season,
         SUM(r.realized_points) AS pick_points_side
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx_fully_realized USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id, r.side_roster_id, r.season
),
pick_points_all AS (
  SELECT r.league_id, r.transaction_id, r.season,
         SUM(r.realized_points) AS pick_points_all
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx_fully_realized USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id, r.season
),
side_season AS (
  SELECT
    COALESCE(ps.league_id, pa.league_id, ks.league_id, ka.league_id) AS league_id,
    COALESCE(ps.transaction_id, pa.transaction_id, ks.transaction_id, ka.transaction_id) AS transaction_id,
    ps.side_roster_id AS side_roster_id,
    COALESCE(ps.season, pa.season, ks.season, ka.season) AS season,
    COALESCE(ps.player_points_side, 0.0) AS player_points_side,
    COALESCE(pa.player_points_all,  0.0) AS player_points_all,
    COALESCE(ks.pick_points_side,   0.0) AS pick_points_side,
    COALESCE(ka.pick_points_all,    0.0) AS pick_points_all
  FROM post_player_points_side ps
  FULL OUTER JOIN post_player_points_all pa
    ON ps.league_id=pa.league_id AND ps.transaction_id=pa.transaction_id AND ps.season=pa.season
  FULL OUTER JOIN pick_points_side ks
    ON COALESCE(ps.league_id, pa.league_id)=ks.league_id
   AND COALESCE(ps.transaction_id, pa.transaction_id)=ks.transaction_id
   AND COALESCE(ps.season, pa.season)=ks.season
  FULL OUTER JOIN pick_points_all ka
    ON COALESCE(ps.league_id, pa.league_id)=ka.league_id
   AND COALESCE(ps.transaction_id, pa.transaction_id)=ka.transaction_id
   AND COALESCE(ps.season, pa.season)=ka.season
),
side_season_net AS (
  SELECT *,
         (player_points_side - (player_points_all - player_points_side)) AS player_net,
         (pick_points_side   - (pick_points_all   - pick_points_side))   AS pick_net
  FROM side_season
),
side_season_loss AS (
  SELECT *,
         (player_net + pick_net) AS net_total,
         CASE WHEN (player_net + pick_net) < 0 THEN -(player_net + pick_net) ELSE 0.0 END AS season_points_lost
  FROM side_season_net
),
points_per_win AS (
  SELECT f.league_id, f.season,
         percentile_approx(CASE WHEN f.points_for > f.points_against THEN f.points_for END, 0.5) AS win_median_pf,
         percentile_approx(CASE WHEN f.points_for < f.points_against THEN f.points_for END, 0.5) AS loss_median_pf
  FROM sleeper_core.fact_team_week f
  JOIN league_filter USING (league_id)
  GROUP BY f.league_id, f.season
),
ppw AS (
  SELECT league_id, season,
         CASE WHEN win_median_pf IS NOT NULL AND loss_median_pf IS NOT NULL AND win_median_pf > loss_median_pf
              THEN (win_median_pf - loss_median_pf) ELSE NULL END AS pts_per_win
  FROM points_per_win
),
side_season_wl AS (
  SELECT s.*,
         p.pts_per_win,
         CASE WHEN p.pts_per_win IS NOT NULL AND p.pts_per_win > 0
              THEN s.season_points_lost / p.pts_per_win ELSE NULL END AS season_wins_lost
  FROM side_season_loss s
  LEFT JOIN ppw p
    ON s.league_id=p.league_id AND s.season=p.season
),
loser_by_trade AS (
  SELECT league_id, transaction_id, side_roster_id,
         SUM(season_points_lost) AS total_points_lost,
         SUM(COALESCE(season_wins_lost,0)) AS total_est_wins_lost
  FROM side_season_wl
  GROUP BY league_id, transaction_id, side_roster_id
),
winner_loser AS (
  SELECT a.league_id, a.transaction_id,
         FIRST(side_roster_id)  FILTER (WHERE rn = 1) AS loser_roster_id,
         FIRST(total_points_lost) FILTER (WHERE rn = 1) AS loser_total_points_lost,
         FIRST(total_est_wins_lost) FILTER (WHERE rn = 1) AS loser_total_wins_lost
  FROM (
    SELECT league_id, transaction_id, side_roster_id, total_points_lost, total_est_wins_lost,
           ROW_NUMBER() OVER (PARTITION BY league_id, transaction_id ORDER BY total_points_lost DESC) AS rn
    FROM loser_by_trade
  ) a
  GROUP BY a.league_id, a.transaction_id
),
loser_ident AS (
  SELECT l.league_id, l.transaction_id, l.loser_roster_id,
         COALESCE(dmr.manager_display_name, CONCAT('Roster ', CAST(l.loser_roster_id AS STRING))) AS loser_manager
  FROM winner_loser l
  LEFT JOIN sleeper_core.dim_manager_roster_map dmr
    ON l.league_id = dmr.league_id AND l.loser_roster_id = dmr.roster_id
),
tx_time AS (
  SELECT t.league_id, t.transaction_id, MAX(to_timestamp(t.created/1000.0)) AS trade_time
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN league_filter USING (league_id)
  WHERE t.type = 'trade'
  GROUP BY t.league_id, t.transaction_id
)
SELECT
  w.transaction_id AS trade_id,
  li.loser_manager  AS loser_manager,
  ROUND(w.loser_total_points_lost, 1) AS total_points_lost,
  ROUND(w.loser_total_wins_lost, 2)   AS est_wins_lost,
  tx.trade_time
FROM winner_loser w
LEFT JOIN loser_ident li ON w.league_id = li.league_id AND w.transaction_id = li.transaction_id
LEFT JOIN tx_time tx ON w.league_id = tx.league_id AND w.transaction_id = tx.transaction_id
ORDER BY total_points_lost DESC
LIMIT __TOPK__
"""

qt = qt.replace("__LEAGUE_NAME__", league_name.replace("'", "''")).replace("__TOPK__", str(top_k))
top_df = spark.sql(qt)
display(top_df)

## 📆 Per‑Season Melt‑Down (for Top Trades)

In [0]:
trade_ids = [row.trade_id for row in top_df.collect()]
trade_csv = ", ".join([f"'{t}'" for t in trade_ids]) if trade_ids else "''"

qs = """
WITH league_filter AS (
  SELECT league_id, name
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
  WHERE name = '__LEAGUE_NAME__'
),
tx AS (
  SELECT DISTINCT t.league_id, t.transaction_id
  FROM workspace.sleeper_raw.sleeper_transactions_snapshot t
  JOIN league_filter USING (league_id)
  WHERE t.type = 'trade'
),
tx_picks AS (
  SELECT b.league_id, b.transaction_id, COUNT(*) AS picks_expected
  FROM sleeper_trades.bridge_trade_pick_mapping_complete b
  JOIN tx USING (league_id, transaction_id)
  GROUP BY b.league_id, b.transaction_id
),
tx_picks_realized AS (
  SELECT r.league_id, r.transaction_id, COUNT(*) AS picks_realized
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id
),
tx_fully_realized AS (
  SELECT t.league_id, t.transaction_id
  FROM tx t
  LEFT JOIN tx_picks p USING (league_id, transaction_id)
  LEFT JOIN tx_picks_realized pr USING (league_id, transaction_id)
  WHERE COALESCE(p.picks_expected, 0) = COALESCE(pr.picks_realized, 0)
),
incoming_players AS (
  SELECT a.league_id, a.transaction_id, a.side_roster_id,
         st.season AS trade_season, st.week AS trade_week, a.player_id
  FROM sleeper_trades.fact_trade_assets_player_only a
  JOIN sleeper_trades.stg_trade_transactions st
    ON a.league_id=st.league_id AND a.transaction_id=st.transaction_id
  JOIN tx_fully_realized USING (league_id, transaction_id)
  WHERE a.direction='incoming'
),
post_player_points_side AS (
  SELECT ip.league_id, ip.transaction_id, ip.side_roster_id,
         fpw.season,
         SUM(fpw.points) AS player_points_side
  FROM incoming_players ip
  JOIN sleeper_core.fact_player_week fpw
    ON fpw.league_id=ip.league_id AND fpw.player_id=ip.player_id AND fpw.roster_id=ip.side_roster_id
   AND ( (fpw.season = ip.trade_season AND fpw.week > ip.trade_week) OR (CAST(fpw.season AS INT) > CAST(ip.trade_season AS INT)) )
  GROUP BY ip.league_id, ip.transaction_id, ip.side_roster_id, fpw.season
),
post_player_points_all AS (
  SELECT ip.league_id, ip.transaction_id, fpw.season,
         SUM(fpw.points) AS player_points_all
  FROM incoming_players ip
  JOIN sleeper_core.fact_player_week fpw
    ON fpw.league_id=ip.league_id AND fpw.player_id=ip.player_id
   AND ( (fpw.season = ip.trade_season AND fpw.week > ip.trade_week) OR (CAST(fpw.season AS INT) > CAST(ip.trade_season AS INT)) )
  GROUP BY ip.league_id, ip.transaction_id, fpw.season
),
pick_points_side AS (
  SELECT r.league_id, r.transaction_id, r.side_roster_id, r.season,
         SUM(r.realized_points) AS pick_points_side
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx_fully_realized USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id, r.side_roster_id, r.season
),
pick_points_all AS (
  SELECT r.league_id, r.transaction_id, r.season,
         SUM(r.realized_points) AS pick_points_all
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx_fully_realized USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id, r.season
),
side_season AS (
  SELECT
    COALESCE(ps.league_id, pa.league_id, ks.league_id, ka.league_id) AS league_id,
    COALESCE(ps.transaction_id, pa.transaction_id, ks.transaction_id, ka.transaction_id) AS transaction_id,
    ps.side_roster_id AS side_roster_id,
    COALESCE(ps.season, pa.season, ks.season, ka.season) AS season,
    COALESCE(ps.player_points_side, 0.0) AS player_points_side,
    COALESCE(pa.player_points_all,  0.0) AS player_points_all,
    COALESCE(ks.pick_points_side,   0.0) AS pick_points_side,
    COALESCE(ka.pick_points_all,    0.0) AS pick_points_all
  FROM post_player_points_side ps
  FULL OUTER JOIN post_player_points_all pa
    ON ps.league_id=pa.league_id AND ps.transaction_id=pa.transaction_id AND ps.season=pa.season
  FULL OUTER JOIN pick_points_side ks
    ON COALESCE(ps.league_id, pa.league_id)=ks.league_id
   AND COALESCE(ps.transaction_id, pa.transaction_id)=ks.transaction_id
   AND COALESCE(ps.season, pa.season)=ks.season
  FULL OUTER JOIN pick_points_all ka
    ON COALESCE(ps.league_id, pa.league_id)=ka.league_id
   AND COALESCE(ps.transaction_id, pa.transaction_id)=ka.transaction_id
   AND COALESCE(ps.season, pa.season)=ka.season
),
side_season_net AS (
  SELECT *,
         (player_points_side - (player_points_all - player_points_side)) AS player_net,
         (pick_points_side   - (pick_points_all   - pick_points_side))   AS pick_net
  FROM side_season
),
side_season_loss AS (
  SELECT *,
         (player_net + pick_net) AS net_total,
         CASE WHEN (player_net + pick_net) < 0 THEN -(player_net + pick_net) ELSE 0.0 END AS season_points_lost
  FROM side_season_net
)
SELECT
  transaction_id AS trade_id,
  season,
  ROUND(season_points_lost, 1) AS season_points_lost
FROM side_season_loss
WHERE transaction_id IN (__TRADE_IDS__)
ORDER BY trade_id, season
"""

qs = qs.replace("__TRADE_IDS__", trade_csv).replace("__LEAGUE_NAME__", league_name.replace("'", "''"))
season_df = spark.sql(qs)
display(season_df)

## 📊 Big Sad Bars & Tiny Teardrops

In [0]:
import matplotlib.pyplot as plt
pdf = top_df.toPandas()
if not pdf.empty:
    fig = plt.figure()
    ax = plt.gca()
    ax2 = ax.twinx()
    ax.bar(pdf["trade_id"], pdf["total_points_lost"])
    ax.set_xlabel("Trade ID")
    ax.set_ylabel("Total Points Lost")
    ax2.plot(pdf["trade_id"], pdf["est_wins_lost"], marker="o")
    ax2.set_ylabel("Estimated Wins Lost")
    plt.title("All‑Time Worst Trades — Sadness Index")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    display(fig)
else:
    print("No data to plot. Everyone traded fairly. Suspicious. 🤨")

## 📈 Per‑Season Catastrophe (Stack of Regrets)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
spdf = season_df.toPandas()
if not spdf.empty:
    pivot = spdf.pivot_table(index="trade_id", columns="season", values="season_points_lost", aggfunc="sum").fillna(0)
    pivot = pivot.sort_index(axis=1)
    fig = plt.figure()
    ax = plt.gca()
    x = range(len(pivot.index))
    width = 0.8 / max(1, len(pivot.columns))
    for i, col in enumerate(pivot.columns):
        ax.bar([xi + i*width for xi in x], pivot[col].values, width=width, label=str(col))
    ax.set_xticks([xi + (len(pivot.columns)-1)*width/2 for xi in x])
    ax.set_xticklabels(pivot.index, rotation=45, ha="right")
    ax.set_xlabel("Trade ID")
    ax.set_ylabel("Points Lost (Season)")
    ax.set_title("Per‑Season Regret (Grouped Bars)")
    ax.legend(title="Season")
    plt.tight_layout()
    display(fig)
else:
    print("No per‑season data. Did everyone read a trading guide? 📈🧐")

## 🧩 What Was Traded? (Players + Picks + Who They Became)

In [0]:
trade_ids = [row.trade_id for row in top_df.collect()]
trade_csv = ", ".join([f"'{t}'" for t in trade_ids]) if trade_ids else "''"

asset_sql = r"""
WITH league_filter AS (
  SELECT league_id
  FROM workspace.sleeper_raw.sleeper_league_info_snapshot
  WHERE name = '__LEAGUE_NAME__'
),

/* Limit to the trades visible in the top table */
tx AS (
  SELECT DISTINCT league_id, transaction_id
  FROM sleeper_trades.stg_trade_transactions
  WHERE league_id IN (SELECT league_id FROM league_filter)
    AND transaction_id IN (__TRADE_IDS__)
),

/* Trade timing — season & week used for "post-trade" window */
trade_time AS (
  SELECT
    t.league_id,
    t.transaction_id,
    MAX(CAST(t.season AS INT)) AS trade_season,
    MAX(CAST(t.week   AS INT)) AS trade_week
  FROM sleeper_trades.stg_trade_transactions t
  JOIN tx USING (league_id, transaction_id)
  GROUP BY t.league_id, t.transaction_id
),

/* ------------------------------
   INCOMING PLAYERS (GOLD table)
   ------------------------------ */
incoming_players AS (
  SELECT a.league_id, a.transaction_id, a.side_roster_id, a.player_id
  FROM sleeper_trades.fact_trade_assets_player_only a
  JOIN tx USING (league_id, transaction_id)
  WHERE a.direction = 'incoming'
),

/* Robust player dims: try core first, then core; fallback to id/UNK */
players_dim AS (
  SELECT
    COALESCE(dp_g.player_id, dp_c.player_id) AS player_id,
    COALESCE(dp_g.full_name, dp_c.full_name) AS full_name,
    COALESCE(dp_g.position,  dp_c.position, 'UNK') AS position
  FROM sleeper_core.dim_players dp_g
  FULL OUTER JOIN sleeper_core.dim_players dp_c
    ON dp_g.player_id = dp_c.player_id
),

players_labeled AS (
  SELECT
    ip.league_id,
    ip.transaction_id,
    ip.side_roster_id,
    'player' AS asset_type,
    COALESCE(pd.full_name, CONCAT('Player ', ip.player_id)) AS asset_label,
    pd.full_name  AS player_name,
    pd.position   AS position,
    ip.player_id  AS player_id,
    CAST(NULL AS STRING) AS realized_season
  FROM incoming_players ip
  LEFT JOIN players_dim pd
    ON pd.player_id = ip.player_id
),

/* ------------------------------------
   REALIZED PICKS (GOLD + optional R/P)
   ------------------------------------ */
realized_picks AS (
  SELECT
    r.league_id,
    r.transaction_id,
    r.side_roster_id,
    r.season AS realized_season,
    r.player_id,
    b.round,
    b.overall_pick
  FROM sleeper_trades.fact_trade_pick_realization r        -- now includes player_id
  JOIN tx USING (league_id, transaction_id)
  LEFT JOIN sleeper_trades.bridge_trade_pick_mapping_complete b
    ON  r.league_id      = b.league_id
    AND r.transaction_id = b.transaction_id
    AND r.side_roster_id = b.to_roster_id
    AND r.season         = b.realized_season
),

picks_labeled AS (
  SELECT
    rp.league_id,
    rp.transaction_id,
    rp.side_roster_id,
    'pick' AS asset_type,
    CASE
      WHEN pd.full_name IS NOT NULL AND rp.round IS NOT NULL AND rp.overall_pick IS NOT NULL
        THEN CONCAT('Round ', rp.round, ' Pick ', rp.overall_pick, ' → ', pd.full_name)
      WHEN pd.full_name IS NOT NULL
        THEN CONCAT('Pick → ', pd.full_name)
      ELSE 'Pick (realized)'
    END AS asset_label,
    pd.full_name  AS player_name,
    pd.position   AS position,
    rp.player_id  AS player_id,
    CAST(rp.realized_season AS STRING) AS realized_season
  FROM realized_picks rp
  LEFT JOIN players_dim pd
    ON pd.player_id = rp.player_id
),

/* Unified assets */
assets AS (
  SELECT * FROM players_labeled
  UNION ALL
  SELECT * FROM picks_labeled
),

/* -----------------------------
   Points after trade
   ----------------------------- */
player_points AS (
  SELECT
    a.league_id, a.transaction_id, a.side_roster_id, a.player_id,
    SUM(fpw.points) AS points_after_trade
  FROM assets a
  JOIN trade_time tw
    ON a.league_id = tw.league_id
   AND a.transaction_id = tw.transaction_id
  JOIN sleeper_core.fact_player_week fpw
    ON fpw.league_id = a.league_id
   AND fpw.player_id = a.player_id
   AND fpw.roster_id = a.side_roster_id
   AND (
      (CAST(fpw.season AS INT) = tw.trade_season AND fpw.week > tw.trade_week) OR
      (CAST(fpw.season AS INT) > tw.trade_season)
   )
  WHERE a.asset_type = 'player'
  GROUP BY a.league_id, a.transaction_id, a.side_roster_id, a.player_id
),

pick_points AS (
  SELECT
    r.league_id, r.transaction_id, r.side_roster_id, CAST(r.season AS STRING) AS realized_season,
    SUM(COALESCE(r.realized_points, 0.0)) AS points_after_trade
  FROM sleeper_trades.fact_trade_pick_realization r
  JOIN tx USING (league_id, transaction_id)
  GROUP BY r.league_id, r.transaction_id, r.side_roster_id, r.season
)

SELECT
  a.transaction_id           AS trade_id,
  dmr.manager_display_name   AS manager_name,
  a.side_roster_id,
  a.asset_type,
  a.asset_label,             -- players: name | picks: "Round R Pick P → Name" | fallback present
  a.player_name,
  a.position,
  a.realized_season,
  CASE
    WHEN a.asset_type='player' THEN COALESCE(pp.points_after_trade, 0.0)
    ELSE COALESCE(pk.points_after_trade, 0.0)
  END AS points_after_trade
FROM assets a
LEFT JOIN player_points pp
  ON a.asset_type='player'
 AND a.league_id=pp.league_id
 AND a.transaction_id=pp.transaction_id
 AND a.side_roster_id=pp.side_roster_id
 AND a.player_id=pp.player_id
LEFT JOIN pick_points pk
  ON a.asset_type='pick'
 AND a.league_id=pk.league_id
 AND a.transaction_id=pk.transaction_id
 AND a.side_roster_id=pk.side_roster_id
 AND a.realized_season=pk.realized_season
LEFT JOIN sleeper_core.dim_manager_roster_map dmr
  ON a.league_id=dmr.league_id AND a.side_roster_id=dmr.roster_id
ORDER BY trade_id, side_roster_id, asset_type DESC, points_after_trade DESC
"""

asset_sql = asset_sql.replace("__LEAGUE_NAME__", league_name.replace("'", "''")).replace("__TRADE_IDS__", trade_csv)
asset_df = spark.sql(asset_sql)
display(asset_df)


## 🍿 Visual: Damage by Asset (Worst Single Trade)

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

if asset_df.count() > 0:
    worst_trade = [row.trade_id for row in top_df.orderBy('total_points_lost', ascending=False).limit(1).collect()]
    if worst_trade:
        tid = worst_trade[0]
        sub = asset_df.filter(asset_df.trade_id == tid).toPandas()
        if not sub.empty:
            sub = sub.sort_values('points_after_trade', ascending=False)
            fig = plt.figure()
            ax = plt.gca()
            ax.bar(sub['asset_name'], sub['points_after_trade'])
            plt.xticks(rotation=45, ha='right')
            ax.set_ylabel('Points After Trade')
            ax.set_title(f'Assets Received — Points After Trade (Trade {tid})')
            plt.tight_layout()
            display(fig)
        else:
            print("No asset rows for the top trade. The chaos is shy today 🤷")
    else:
        print("No trades found. Everyone is a perfect economist now. 📈")
else:
    print("No asset data. Check upstream tables.")

---

## 🥳 Done!
- Share screenshots. Add spicy commentary. Start a group chat riot.
- Want a **clean SQL dashboard** version with dropdowns? Ping your friendly data clown (me). 🎪
